# Lab 04 : Portes Qualité Silver, Imputation & Contrat de Schéma Final
**Cours :** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format :** Lab #4 (EN : Jeu 26.11.2026 / FR : Ven 27.11.2026) · 3 Heures (180 min)

---

## 🎯 1. Contexte & Architecture-as-Code
Dans le Lab 03, vous avez normalisé les clés, converti les dates et supprimé les doublons dans `silver_part1/*.parquet` tout en conservant les anomalies visibles. Aujourd'hui, vous finalisez toute la **couche Silver** sur le **Cas 0 (`EuroPack Beverage Bottling`)**, un fichier et une étape ciblée à la fois (**Problème → Découverte → Solution**) :

*(Note : les jointures multi-tables, la cardinalité et le piège Fan-Out `480.0 min` sont traités directement en SQL dans le **Lab 05 : Modélisation en étoile DuckDB** !)*

```mermaid
flowchart TD
    S1["2. Charger silver_part1/*.parquet (2.1)"] --> O0["3. Valeurs aberrantes : IQR vs 3-Sigma & marquage actif (3.1 - 3.3)"]
    O0 --> Q0["4. Porte DAMA 1 : Quarantaine de lignes vs Mise à NaN capteur (4.1 - 4.4)"]
    Q0 --> I0["5. Interpolation (5.1 - 5.2) -> delta_temp_C (5.3) -> Contrôle PII (5.4)"]
    I0 --> C0["6. Contrat schéma (6.1) -> Sauvegarde (6.2) -> Audit DuckDB (6.3)"]
```

**Vos objectifs :**
1. **Comparer IQR vs 3-Sigma & utiliser IQR (Section 3) :** Constater que `-99.0` / `85.0 bar` élargissent 3-Sigma à `[-8.06, 16.26] bar`, examiner les `2 984` lignes détectées par **IQR uniquement** (vrais arrêts `0.0 bar` et surpressions réelles `5.7–8.4 bar` à ne PAS supprimer !), et appliquer l'IQR via le drapeau `is_pressure_iqr_outlier`.
2. **Porte Qualité DAMA à deux niveaux (Section 4) :** Séparer les **lignes irrécupérables** (`DQR-101, 102, 104, 105` — 11 lignes mises en quarantaine et retirées) des **glitches capteur isolés** (`DQR-103` — 40 lignes de pression portant `34 395` vraies bouteilles : journalisées, mises à `NaN` et réparées par interpolation !).
3. **Interpolation temporelle, Feature Engineering & contrôle PII (Section 5) :** Réparer `pasteurizer_temp_C` (`120` trous) et `filler_pressure_bar` (`40` pics mis à `NaN`) via `.interpolate(method='time')` par `line_id` (`142 704` bouteilles préservées !), calculer `delta_temp_C` et vérifier zéro fuite PII sur les 5 tables Silver via `set.isdisjoint()`.
4. **Contrat de schéma Silver final & audit PyArrow/DuckDB (Section 6) :** Imposer `SILVER_SCHEMAS` + horodatages UTC, exporter `silver/*.parquet` et auditer Bronze vs Silver final avec PyArrow & DuckDB `DESCRIBE`.

## ⚙️ 2. Configuration de l'environnement & chargement de `silver_part1`
### 2.1 Charger les 5 tables `silver_part1` & vérifier l'état de départ

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
P1_DIR = resolve_stage_dir("silver_part1")
SILVER_OUT = SANDBOX_DATA / "silver"
SILVER_OUT.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(P1_DIR / "silver_part1_telemetry.parquet")
df_stop = pd.read_parquet(P1_DIR / "silver_part1_stoppages.parquet")
df_lines = pd.read_parquet(P1_DIR / "silver_lines.parquet")
df_shifts = pd.read_parquet(P1_DIR / "silver_shifts.parquet")
df_formats = pd.read_parquet(P1_DIR / "silver_bottle_formats.parquet")

print("Silver Part 1 source :", P1_DIR)
print("Silver Final output  :", SILVER_OUT)
print(
    f"Loaded 5 Silver Part 1 tables -> "
    f"telemetry: {len(df_telem):,} rows (timestamp={df_telem['timestamp'].dtype}) | "
    f"stoppages: {len(df_stop):,} logs | lines: {len(df_lines)} | shifts: {len(df_shifts)} | formats: {len(df_formats)}"
)


## 📊 3. Analyse des valeurs aberrantes — IQR de Tukey vs 3-Sigma gaussien (`df_telem`)
### 3.1 Découverte — Inspecter les distributions & pics extrêmes
**Le problème :** Avant de choisir une règle statistique, inspectez les statistiques résumées (`mean`, `std`, `min`, `50%`, `max`) de `filler_pressure_bar` et `pasteurizer_temp_C`. Observez comment 40 pics extrêmes (`-99.0` et `85.0 bar`) gonflent l'écart-type `std` à `4.05 bar` !

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df[cols].describe()` | Statistiques résumées (`mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) | `df_telem[["filler_pressure_bar", "pasteurizer_temp_C"]].describe()` |
> | `df.loc[mask, cols]` | Filtrer les lignes selon un masque booléen | `df_telem.loc[(df_telem["filler_pressure_bar"] < 0) \| (df_telem["filler_pressure_bar"] > 12)]` |

In [ ]:
print("=== Summary Statistics (notice how -99.0 / 85.0 bar inflate filler_pressure_bar std!) ===")
print(df_telem[["filler_pressure_bar", "pasteurizer_temp_C"]].describe().round(2).to_string())

extreme_p = df_telem.loc[(df_telem["filler_pressure_bar"] < 0.0) | (df_telem["filler_pressure_bar"] > 12.0)]
print(f"\nExtreme pressure readings outside physical [0.0, 12.0] bar: {len(extreme_p)} rows")
print(extreme_p[["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(5).to_string(index=False))


### 3.2 Solution — Comparer IQR vs 3-Sigma & inspecter les lignes qui diffèrent
**L'enseignement clé :** Implémentez les deux estimateurs et inspectez **l'ensemble de désaccord** (lignes signalées par IQR uniquement vs 3-Sigma uniquement) :
- **3-Sigma (`mean ± 3*std`) :** S'élargit à `[-8.06, 16.26] bar` — tellement déformé qu'il tolère des pressions négatives impossibles et ne détecte que les 40 pics extrêmes (`0` ligne détectée par 3-Sigma seul !).
- **IQR de Tukey (`Q1 - 1.5*IQR` .. `Q3 + 1.5*IQR`) :** Reste ancré à `[3.10, 5.42] bar` et repère `3 024` lignes (`2 984` manquées par 3-Sigma !).
- **Pourquoi INSPECTER avant de supprimer :** Ces `2 984` lignes sont de **vrais arrêts ligne (`0.00 bar`, `bottled_units = 0`)** et de **vraies surpressions d'usure (`5.7–8.4 bar`)** avant panne ! En industrie, on utilise les **bornes physiques (`[0, 12] bar`)** pour les pannes capteur, et l'**IQR** pour **marquer** les anomalies statistiques (jamais supprimer aveuglément !).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.dropna().quantile(q)` | Calculer Q1 (`0.25`) et Q3 (`0.75`) | `q1, q3 = valid.quantile(0.25), valid.quantile(0.75)` |
> | `Series.mean()` / `Series.std()` | Moyenne et écart-type | `mu, sd = valid.mean(), valid.std()` |
> | `mask_iqr & ~mask_s3` | Lignes signalées par IQR **uniquement** (manquées par 3-Sigma) | `df_telem.loc[mask_iqr & ~mask_s3]` |

In [ ]:
def get_iqr_bounds(s: pd.Series, k: float = 1.5):
    # TODO: Drop NaNs, compute Q1=quantile(0.25), Q3=quantile(0.75), iqr=Q3-Q1, return (float(q1 - k*iqr), float(q3 + k*iqr))
    return None

def get_3sigma_bounds(s: pd.Series):
    # TODO: Drop NaNs, compute mu=mean(), sd=std(), return (float(mu - 3*sd), float(mu + 3*sd))
    return None

if get_iqr_bounds(df_telem["filler_pressure_bar"]) is not None and get_3sigma_bounds(df_telem["filler_pressure_bar"]) is not None:
    rows = []
    for col in ("filler_pressure_bar", "pasteurizer_temp_C"):
        iqr_lo, iqr_hi = get_iqr_bounds(df_telem[col])
        s3_lo, s3_hi = get_3sigma_bounds(df_telem[col])
        m_iqr = (df_telem[col] < iqr_lo) | (df_telem[col] > iqr_hi)
        m_s3 = (df_telem[col] < s3_lo) | (df_telem[col] > s3_hi)
        rows.append({
            "column": col,
            "IQR_bounds": f"[{iqr_lo:.2f}, {iqr_hi:.2f}]",
            "IQR_flagged": int(m_iqr.sum()),
            "3Sigma_bounds": f"[{s3_lo:.2f}, {s3_hi:.2f}]",
            "3Sigma_flagged": int(m_s3.sum()),
            "IQR_only": int((m_iqr & ~m_s3).sum()),
            "3Sigma_only": int((m_s3 & ~m_iqr).sum()),
        })
    print("=== IQR vs 3-SIGMA COMPARISON (RAW TELEMETRY) ===")
    print(pd.DataFrame(rows).to_string(index=False))

    p_iqr_lo, p_iqr_hi = get_iqr_bounds(df_telem["filler_pressure_bar"])
    p_s3_lo, p_s3_hi = get_3sigma_bounds(df_telem["filler_pressure_bar"])
    iqr_only_p = df_telem.loc[
        ((df_telem["filler_pressure_bar"] < p_iqr_lo) | (df_telem["filler_pressure_bar"] > p_iqr_hi))
        & df_telem["filler_pressure_bar"].between(p_s3_lo, p_s3_hi)
    ]
    print(f"\nSample of the {len(iqr_only_p):,} pressure rows flagged by IQR ONLY (real 0.00 bar stoppages & 5.7-8.4 bar stress!):")
    print(iqr_only_p[["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(6).to_string(index=False))
else:
    print("⏳ [TODO] Implement get_iqr_bounds() and get_3sigma_bounds() above.")


### 3.3 Solution — Mettre l'IQR en pratique : marquer les anomalies de pression en production active
**L'usage concret de l'IQR :** En excluant les périodes d'arrêt (`bottled_units == 0`) et les glitches hors `[0, 12] bar`, l'IQR de Tukey sur la **production active** donne une plage normale précise (`[3.22, 5.38] bar`). Au lieu de supprimer les pics de contrainte, ajoutez un drapeau booléen **`is_pressure_iqr_outlier`** pour l'analyse Maintenance sans perdre une seule bouteille !

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.between(lo, hi)` | Test d'intervalle inclusif (`lo <= x <= hi`) | `df_telem["filler_pressure_bar"].between(0.0, 12.0)` |
> | `active_mask & ~s.between(lo, hi)` | Marquer les écarts statistiques uniquement sur lignes actives valides | `df_telem["is_pressure_iqr_outlier"] = active_valid & ~df_telem["filler_pressure_bar"].between(act_lo, act_hi)` |

In [ ]:
active_valid = df_telem["filler_pressure_bar"].between(0.0, 12.0) & (df_telem["bottled_units"] > 0)

if get_iqr_bounds(df_telem["filler_pressure_bar"]) is not None and get_3sigma_bounds(df_telem["filler_pressure_bar"]) is not None:
    act_iqr_lo, act_iqr_hi = get_iqr_bounds(df_telem.loc[active_valid, "filler_pressure_bar"])
    act_s3_lo, act_s3_hi = get_3sigma_bounds(df_telem.loc[active_valid, "filler_pressure_bar"])

    # TODO: Create boolean column df_telem["is_pressure_iqr_outlier"] = active_valid & ~df_telem["filler_pressure_bar"].between(act_iqr_lo, act_iqr_hi)
    if "is_pressure_iqr_outlier" in df_telem.columns:
        n_iqr_act = int(df_telem["is_pressure_iqr_outlier"].sum())
        n_s3_act = int((active_valid & ~df_telem["filler_pressure_bar"].between(act_s3_lo, act_s3_hi)).sum())
        print(f"Active running pressure bounds -> IQR: [{act_iqr_lo:.2f}, {act_iqr_hi:.2f}] ({n_iqr_act:,} flagged) | 3-Sigma: [{act_s3_lo:.2f}, {act_s3_hi:.2f}] ({n_s3_act:,} flagged)")
        print("Tagged `is_pressure_iqr_outlier` without deleting any valid production rows!")
    else:
        print("⏳ [TODO] Assign df_telem['is_pressure_iqr_outlier'] above.")
else:
    print("⏳ [TODO] Complete Step 3.2 first.")


## 🚨 4. Porte Qualité DAMA 1 — Quarantaine de ligne vs Mise à `NaN` capteur
Toutes les anomalies ne doivent pas subir le même traitement ! On distingue deux niveaux :
1. **Erreurs de ligne irrécupérables (`QUARANTINED_ROW`) :** Actif inconnu (`LINE_Z`), date future (`2099`), ou compteurs corrompus (`rejected_units > bottled_units`). Ces lignes doivent être **retirées** de Silver et archivées dans `quarantine_errors.parquet`.
2. **Glitches capteur isolés (`NULLIFIED_AND_INTERPOLATED`) :** Une sonde affichant `-99.0` ou `85.0 bar` sur une ligne dont `bottled_units` est valide. On journalise l'incident, on passe uniquement `filler_pressure_bar` à `NaN` et on répare en Section 5.

### 4.1 Découverte (Fichier 1 : `df_telem`) — Repérer les lignes irrécupérables (`DQR-101, 102, 104`)
**Le problème :** Identifiez les trois catégories d'erreurs irrécupérables sur `df_telem` (en lecture seule d'abord pour contrôler ce qui sera retiré) :
- `DQR-101` (`Consistency`) : `line_id` orphelin absent de `valid_lines` (`LINE_Z`)
- `DQR-102` (`Timeliness`) : `timestamp` nul (`NaT`) ou futur (`> 2027-01-01`)
- `DQR-104` (`Validity`) : Quantités négatives ou `rejected_units > bottled_units`

> **Mémo syntaxe Pandas :**
> | Méthode / Opérateur | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `~df["line_id"].isin(valid_lines)` | Repérer les clés absentes du référentiel ERP | `q_orphan = ~df_telem["line_id"].isin(valid_lines)` |
> | `pd.Timestamp("2027-01-01")` | Seuil natif pour comparer des timestamps | `df_telem["timestamp"] > pd.Timestamp("2027-01-01")` |
> | `cond_a \| cond_b` | OU logique bit-à-bit entre masques booléens | `(df_telem["bottled_units"] < 0) \| (df_telem["rejected_units"] > df_telem["bottled_units"])` |

In [ ]:
valid_lines = set(df_lines["line_id"])

# TODO (Discovery — Read-Only): Build boolean masks for DQR-101, DQR-102, and DQR-104
q_orphan = None  # ~df_telem["line_id"].isin(valid_lines)
q_future = None  # df_telem["timestamp"].isna() | (df_telem["timestamp"] > pd.Timestamp("2027-01-01"))
q_units  = None  # (df_telem["bottled_units"] < 0) | (df_telem["rejected_units"] < 0) | (df_telem["rejected_units"] > df_telem["bottled_units"])

if all(m is not None for m in (q_orphan, q_future, q_units)):
    print(f"[Discovery] Unrecoverable telemetry rows -> DQR-101: {q_orphan.sum()} | DQR-102: {q_future.sum()} | DQR-104: {q_units.sum()}")
    print("\nPreview of flagged unrecoverable rows:")
    print(df_telem.loc[q_orphan | q_future | q_units, ["timestamp", "line_id", "bottle_format", "bottled_units", "rejected_units"]].to_string(index=False))
else:
    print("⏳ [TODO] Define boolean masks q_orphan, q_future, and q_units above.")


### 4.2 Solution (Fichier 1 : `df_telem`) — Mettre en quarantaine & retirer les lignes irrécupérables
**La correction :** Archivez les lignes signalées avec des masques mutuellement exclusifs (`q_future & ~q_orphan`, etc., pour ne jamais compter deux fois une ligne) et retirez uniquement ces lignes de `df_telem`.

> **Mémo syntaxe Pandas :**
> | Méthode / Opérateur | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `mask_b & ~mask_a` | Masque mutuellement exclusif (évite le double comptage) | `q_future & ~q_orphan` |
> | `df.loc[~bad_mask].copy()` | Conserver uniquement les lignes valides | `df_telem = df_telem.loc[~(q_orphan \| q_future \| q_units)].copy()` |

In [ ]:
# Helper provided: packages flagged rows into a standardized DAMA Audit/Quarantine DataFrame
def extract_quarantine(df: pd.DataFrame, mask: pd.Series, src: str, rid: str, code: str, crit: str, action: str = "QUARANTINED_ROW") -> pd.DataFrame:
    cols = ["dq_rule_id", "error_code", "dama_criterion", "source_table", "action", "raw_record"]
    if not mask.any():
        return pd.DataFrame(columns=cols)
    return pd.DataFrame({
        "dq_rule_id": rid,
        "error_code": code,
        "dama_criterion": crit,
        "source_table": src,
        "action": action,
        "raw_record": df.loc[mask].astype(str).to_dict(orient="records"),
    }).assign(raw_record=lambda d: d["raw_record"].astype(str))

q_telem_parts = []
if all(m is not None for m in (q_orphan, q_future, q_units)):
    q_telem_parts = [
        extract_quarantine(df_telem, q_orphan, "line_telemetry.csv", "DQR-101", "ORPHAN_LINE_ID", "Consistency", "QUARANTINED_ROW"),
        extract_quarantine(df_telem, q_future & ~q_orphan, "line_telemetry.csv", "DQR-102", "INVALID_OR_FUTURE_TIMESTAMP", "Timeliness", "QUARANTINED_ROW"),
        extract_quarantine(df_telem, q_units & ~q_orphan & ~q_future, "line_telemetry.csv", "DQR-104", "NEGATIVE_OR_IMPOSSIBLE_UNITS", "Validity", "QUARANTINED_ROW"),
    ]
    before_n = len(df_telem)
    # TODO: Filter out unrecoverable rows -> df_telem = df_telem.loc[~(q_orphan | q_future | q_units)].copy()
    print(f"[Verification] Removed {before_n - len(df_telem)} unrecoverable rows -> df_telem: {len(df_telem):,} rows.")
else:
    print("⏳ [TODO] Complete Step 4.1 masks first.")


### 4.3 Découverte & Solution (Fichier 1 : `df_telem`) — Passer à `NaN` les glitches de pression (`DQR-103`, garder les `34 395` bouteilles !)
**Pourquoi les bornes physiques sont `[0.0, 12.0] bar` (spécification métier Cas 0) :**
- **Borne basse `0.0 bar` :** La pression relative d'une soutireuse ne peut pas être négative — à l'arrêt, elle vaut légitimement **`0.0 bar`**. Une valeur comme `-99.0 bar` est un code d'erreur capteur physiquement impossible.
- **Borne haute `12.0 bar` :** Le soutirage normal fonctionne à `3.7–5.2 bar` et les vannes usées montent à `5.9–8.39 bar` avant panne. Au-delà de **`12.0 bar`**, les bouteilles PET éclatent et les soupapes de sécurité déclenchent — `85.0 bar` est donc forcément une mesure corrompue.

**Pourquoi mettre à `NaN` au lieu de supprimer la ligne ?** Ces `40` lignes portent **`34 395` vraies bouteilles** ! Journalisez-les (`action="NULLIFIED_AND_INTERPOLATED"`) et passez uniquement `filler_pressure_bar = np.nan` pour les réparer en Section 5.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.loc[mask, "col"].sum()` | Mesurer la production en jeu avant d'agir | `df_telem.loc[q_press, "bottled_units"].sum()` |
> | `df.loc[mask, "col"] = np.nan` | Mettre à `NaN` UNIQUEMENT le capteur fautif (garder la ligne !) | `df_telem.loc[q_press, "filler_pressure_bar"] = np.nan` |

In [ ]:
# STEP A (Discovery): Flag readings outside physical envelope [0.0, 12.0] bar
# TODO: Build q_press = (df_telem["filler_pressure_bar"] < 0.0) | (df_telem["filler_pressure_bar"] > 12.0)
q_press = None
q_press_part = pd.DataFrame()

if q_press is not None:
    press_bottles_saved = int(df_telem.loc[q_press, "bottled_units"].sum())
    print(f"[Discovery] Found {q_press.sum()} impossible pressure readings carrying {press_bottles_saved:,} real bottled_units!")
    print(df_telem.loc[q_press, ["timestamp", "line_id", "bottle_format", "filler_pressure_bar", "bottled_units"]].head(4).to_string(index=False))

    # STEP B (Solution): Log in audit table AND nullify only filler_pressure_bar (keep the rows!)
    q_press_part = extract_quarantine(df_telem, q_press, "line_telemetry.csv", "DQR-103", "PRESSURE_OUT_OF_BOUNDS_0_12", "Accuracy", "NULLIFIED_AND_INTERPOLATED")
    # TODO: df_telem.loc[q_press, "filler_pressure_bar"] = np.nan
    print(f"\n[Verification] Null filler_pressure_bar count ready for Section 5 interpolation: {df_telem['filler_pressure_bar'].isna().sum()}")
else:
    print("⏳ [TODO] Define q_press and nullify filler_pressure_bar above.")


### 4.4 Découverte & Solution (Fichier 2 : `df_stop`) — Quarantaine `DQR-105` & assemblage de `quarantine_df`
**Le problème :** Appliquez la porte de quarantaine au Fichier 2 (`df_stop`, isolant `STOP-20269999`), puis concaténez tous les rejets et réparations dans `quarantine_df` (`11` `QUARANTINED_ROW` + `40` `NULLIFIED_AND_INTERPOLATED` = **`51` événements DQ audités**).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `pd.concat(liste_dfs, ignore_index=True)` | Empiler plusieurs DataFrames avec un index propre `0..N-1` | `quarantine_df = pd.concat([p for p in q_parts if not p.empty], ignore_index=True)` |
> | `df.groupby(["dq_rule_id", "error_code", "action"]).size()` | Compter les incidents par règle et action appliquée | `quarantine_df.groupby(["dq_rule_id", "error_code", "action"]).size()` |

In [ ]:
# STEP A (Discovery): Flag invalid stoppage logs in df_stop
# TODO: stop_bad = (~df_stop["line_code"].isin(valid_lines)) | df_stop["event_date"].isna() | (df_stop["event_date"] > pd.Timestamp("2027-01-01"))
stop_bad = None

quarantine_df = None
if stop_bad is not None and q_telem_parts and not q_press_part.empty:
    print(f"[Discovery] Flagged {stop_bad.sum()} invalid stoppage log(s):")
    print(df_stop.loc[stop_bad, ["log_id", "event_date", "line_code", "stoppage_type", "root_cause", "downtime_min"]].to_string(index=False))

    # STEP B (Solution): Quarantine DQR-105, filter df_stop, and combine all audit slices
    q_stop_part = extract_quarantine(df_stop, stop_bad, "stoppage_logs.json", "DQR-105", "INVALID_STOPPAGE_LOG", "Completeness", "QUARANTINED_ROW")
    df_stop = df_stop.loc[~stop_bad].copy()

    q_parts = [*q_telem_parts, q_press_part, q_stop_part]
    quarantine_df = pd.concat([p for p in q_parts if not p.empty], ignore_index=True)

    print("\n[Solution] DAMA DQ Audit Summary (Quarantined Rows + Nullified Sensor Glitches):")
    print(quarantine_df.groupby(["dq_rule_id", "error_code", "action"]).size().to_string())
    print(f"Total audited DQ events: {len(quarantine_df)} | Clean df_stop rows: {len(df_stop):,}")
else:
    print("⏳ [TODO] Complete Steps 4.1–4.3 and define stop_bad above.")


## 🌡️ 5. Interpolation temporelle, Feature Engineering & contrôle PII
### 5.1 Découverte — Localiser tous les trous capteurs (`pasteurizer_temp_C` & `filler_pressure_bar` mis à `NaN`)
**Le problème :** `df_telem` contient **deux** séries de trous capteurs réparables sur des lignes de production valides :
1. **`120` valeurs `pasteurizer_temp_C` manquantes** (`108 309` vraies bouteilles !)
2. **`40` pics `filler_pressure_bar` mis à `NaN`** (`34 395` vraies bouteilles préservées en 4.3 !)

Triez chronologiquement par `line_id` et observez un exemple de chaque trou avec ses relevés voisins à ±15 minutes.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.sort_values(["line_id", "timestamp"]).reset_index(drop=True)` | Trier chronologiquement par ligne avant interpolation | Indispensable avant `.interpolate(method="time")` |
> | `Series.isna()` | Masque booléen des valeurs `NaN` | `missing_temp = df_telem["pasteurizer_temp_C"].isna()` |
> | `df.loc[mask, "bottled_units"].sum()` | Production réelle sauvée par l'interpolation | `df_telem.loc[missing_temp \| missing_press, "bottled_units"].sum()` |

In [ ]:
df_telem = df_telem.sort_values(["line_id", "timestamp"]).reset_index(drop=True)
missing_temp = df_telem["pasteurizer_temp_C"].isna()
missing_press = df_telem["filler_pressure_bar"].isna()
total_saved_bottles = int(df_telem.loc[missing_temp | missing_press, "bottled_units"].sum())

print(f"Missing pasteurizer_temp_C gaps : {missing_temp.sum():>3} ({int(df_telem.loc[missing_temp, 'bottled_units'].sum()):,} bottles)")
print(f"Nullified filler_pressure_bar   : {missing_press.sum():>3} ({int(df_telem.loc[missing_press, 'bottled_units'].sum()):,} bottles)")
print(f"Total real bottles preserved    : {total_saved_bottles:,} bottles (saved vs dropna()!)")

first_temp_idx = int(missing_temp.idxmax())
print("\nSample pasteurizer_temp_C gap with adjacent 15-min readings:")
print(df_telem.loc[max(0, first_temp_idx - 1):first_temp_idx + 1, ["timestamp", "line_id", "pasteurizer_temp_C", "bottled_units"]].to_string(index=False))

if missing_press.any():
    first_press_idx = int(missing_press.idxmax())
    print("\nSample nullified filler_pressure_bar gap with adjacent 15-min readings:")
    print(df_telem.loc[max(0, first_press_idx - 1):first_press_idx + 1, ["timestamp", "line_id", "filler_pressure_bar", "bottled_units"]].to_string(index=False))
else:
    first_press_idx = None


### 5.2 Solution — Interpolation temporelle par ligne (`method='time'`)
**La correction :** Tracez chaque ligne réparée dans **`is_imputed_flag = missing_temp | missing_press`**, puis interpolez `pasteurizer_temp_C` ET `filler_pressure_bar` linéairement dans le temps (**`method="time"`**) **séparément par `line_id`** afin de ne jamais mélanger deux lignes !

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `g.set_index("timestamp")[col].interpolate(method="time", limit_direction="both")` | Interpolation linéaire pondérée par le temps (`DatetimeIndex`) | Dans `_interp_col(g, col)` par `line_id` |
> | `Series.fillna(interp_vals)` | Remplacer uniquement les `NaN` par les valeurs interpolées | `df_telem[col] = df_telem[col].fillna(interp_vals).round(2)` |
> | `Series.between(0.0, 12.0).all()` | Vérifier que 100 % des pressions sont dans `[0.0, 12.0]` bar | `assert df_telem["filler_pressure_bar"].between(0.0, 12.0).all()` |

In [ ]:
df_telem["is_imputed_flag"] = missing_temp | missing_press

def _interp_col(g: pd.DataFrame, col: str) -> pd.Series:
    s = g.set_index("timestamp")[col].interpolate(method="time", limit_direction="both")
    return pd.Series(s.values, index=g.index)

# TODO: Interpolate both "pasteurizer_temp_C" and "filler_pressure_bar" per line_id:
# for col in ("pasteurizer_temp_C", "filler_pressure_bar"):
#     interp_vals = pd.concat([_interp_col(grp, col) for _, grp in df_telem.groupby("line_id", sort=False)]).sort_index()
#     df_telem[col] = df_telem[col].fillna(interp_vals).round(2)

if df_telem["pasteurizer_temp_C"].isna().sum() == 0 and df_telem["filler_pressure_bar"].isna().sum() == 0:
    assert df_telem["filler_pressure_bar"].between(0.0, 12.0).all(), "Out-of-bounds pressure remains!"
    print("Temperature gap AFTER per-line time interpolation:")
    print(df_telem.loc[max(0, first_temp_idx - 1):first_temp_idx + 1, ["timestamp", "line_id", "pasteurizer_temp_C", "is_imputed_flag"]].to_string(index=False))
    if first_press_idx is not None:
        print("\nPressure spike AFTER nullify + per-line time interpolation:")
        print(df_telem.loc[max(0, first_press_idx - 1):first_press_idx + 1, ["timestamp", "line_id", "filler_pressure_bar", "is_imputed_flag", "bottled_units"]].to_string(index=False))
    print(f"\n[OK] Imputed {int(df_telem['is_imputed_flag'].sum())} sensor gaps ({missing_temp.sum()} temp + {missing_press.sum()} pressure) -> 0 nulls!")
else:
    print("⏳ [TODO] Interpolate pasteurizer_temp_C and filler_pressure_bar per line_id above.")


### 5.3 Solution — Feature Engineering (`delta_temp_C`)
**L'objectif :** Maintenant que `pasteurizer_temp_C` n'a plus aucun `NaN`, calculez l'écart thermique **`delta_temp_C = pasteurizer_temp_C - ambient_temp_C`**, utilisé par la Maintenance au Lab 05 pour détecter la surchauffe avant panne !

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `(df["col_a"] - df["col_b"]).round(2)` | Soustraction vectorisée arrondie à 2 décimales | `df_telem["delta_temp_C"] = (df_telem["pasteurizer_temp_C"] - df_telem["ambient_temp_C"]).round(2)` |

In [ ]:
# TODO: Compute df_telem["delta_temp_C"] = (df_telem["pasteurizer_temp_C"] - df_telem["ambient_temp_C"]).round(2)

if "delta_temp_C" in df_telem.columns and df_telem["delta_temp_C"].isna().sum() == 0:
    print("Derived delta_temp_C summary statistics:")
    print(df_telem["delta_temp_C"].describe().round(2).to_string())
else:
    print("⏳ [TODO] Complete Step 5.2 and compute df_telem['delta_temp_C'] above.")


### 5.4 Vérification — Contrôle de confidentialité / exclusion PII (`set.isdisjoint`)
**Le problème :** Conformément au RGPD et à la gouvernance des données, aucune **donnée personnelle (PII)** — noms, emails, identifiants nationaux ou notes médicales — ne doit passer dans les couches analytiques Silver ou Gold. Contrôlez les colonnes des **5 tables Silver** par rapport à `PII_FORBIDDEN_COLS` avant toute sauvegarde !

> **Mémo syntaxe Python (`set`) :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `set_a.isdisjoint(set_b)` | Renvoie **`True`** si `set_a` et `set_b` n'ont **aucun élément en commun** (intersection vide) ; **`False`** si une colonne interdite existe | `PII_FORBIDDEN_COLS.isdisjoint({c.lower() for c in df.columns})` |
> | `set_a & set_b` | Intersection de deux ensembles — liste les colonnes en fuite pour le message d'erreur | `leaks = PII_FORBIDDEN_COLS & {c.lower() for c in df.columns}` |

In [ ]:
PII_FORBIDDEN_COLS = {
    "first_name", "last_name", "full_name",
    "corporate_email", "national_id_masked", "details_medical_note",
}

silver_tables = {
    "silver_telemetry": df_telem,
    "silver_stoppages": df_stop,
    "silver_lines": df_lines,
    "silver_shifts": df_shifts,
    "silver_bottle_formats": df_formats,
}

for tbl_name, df_tbl in silver_tables.items():
    col_set = {c.lower() for c in df_tbl.columns}
    # set.isdisjoint() returns True ONLY when col_set shares zero columns with PII_FORBIDDEN_COLS
    assert PII_FORBIDDEN_COLS.isdisjoint(col_set), f"PII leakage in {tbl_name}: {PII_FORBIDDEN_COLS & col_set}"
    print(f"  [PASS] {tbl_name:<24} -> 0 PII columns detected ({len(col_set)} columns checked)")


## 💾 6. Contrat de Schéma Silver Final, Sauvegarde & Audit PyArrow / DuckDB
### 6.1 Solution — Typer `_ingestion_timestamp_utc` (UTC) & appliquer `SILVER_SCHEMAS`
**Le problème :** Avant d'écrire sur disque, convertissez `_ingestion_timestamp_utc` en **`timestamp[ns, tz=UTC]`** sur les 5 tables et appliquez le contrat explicite (`string`, `float64`, `int64`, `bool`) sur chaque colonne.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `pd.to_datetime(s, format="ISO8601", utc=True, errors="coerce")` | Convertir l'audit ISO-8601 -> `datetime64[ns, UTC]` | `_df["_ingestion_timestamp_utc"] = pd.to_datetime(...)` |
> | `Series.astype("string")` / `Series.astype("bool")` | Types texte et booléen explicites | `out[col] = out[col].astype("string")` |
> | `pd.to_numeric(s, errors="coerce").astype("float64" / "int64")` | Typage numérique strict | `out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")` |

In [ ]:
SILVER_SCHEMAS = {
    "telemetry": {
        "string": ["line_id", "bottle_format", "shift_code", "_source_file"],
        "float64": ["ambient_temp_C", "pasteurizer_temp_C", "filler_pressure_bar", "delta_temp_C"],
        "int64": ["conveyor_speed_bpm", "valve_cycles_k", "bottled_units", "rejected_units"],
        "bool": ["is_imputed_flag", "is_pressure_iqr_outlier"],
    },
    "stoppages": {
        "string": ["log_id", "line_code", "stoppage_type", "root_cause", "technician_team", "shift_code", "_source_file"],
        "int64": ["downtime_min"],
    },
    "lines": {
        "string": ["line_id", "line_name", "filler_type", "_source_file"],
        "float64": ["nominal_cycle_time_sec", "hourly_operating_cost_eur"],
    },
    "shifts": {
        "string": ["shift_id", "shift_name", "start_time", "end_time", "_source_file"],
        "int64": ["planned_duration_hours"],
    },
    "bottle_formats": {
        "string": ["format_code", "format_name", "packaging_material", "_source_file"],
        "float64": ["unit_reject_cost_eur"],
        "int64": ["target_hourly_output"],
    },
}

def enforce_dtypes(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    out = df.copy()
    for col in spec.get("string", []):
        out[col] = out[col].astype("string")
    for col in spec.get("float64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
    for col in spec.get("int64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("int64")
    for col in spec.get("bool", []):
        out[col] = out[col].astype("bool")
    return out

schema_ready = (
    quarantine_df is not None
    and "delta_temp_C" in df_telem.columns
    and "is_pressure_iqr_outlier" in df_telem.columns
    and df_telem["pasteurizer_temp_C"].isna().sum() == 0
    and df_telem["filler_pressure_bar"].isna().sum() == 0
)

if schema_ready:
    for _df in (df_telem, df_stop, df_lines, df_shifts, df_formats):
        _df["_ingestion_timestamp_utc"] = pd.to_datetime(
            _df["_ingestion_timestamp_utc"], format="ISO8601", utc=True, errors="coerce"
        )
    df_telem = enforce_dtypes(df_telem, SILVER_SCHEMAS["telemetry"])
    df_stop = enforce_dtypes(df_stop, SILVER_SCHEMAS["stoppages"])
    df_lines = enforce_dtypes(df_lines, SILVER_SCHEMAS["lines"])
    df_shifts = enforce_dtypes(df_shifts, SILVER_SCHEMAS["shifts"])
    df_formats = enforce_dtypes(df_formats, SILVER_SCHEMAS["bottle_formats"])
    print("Enforced Silver dtypes on df_telem:")
    print(df_telem.dtypes.to_string())
else:
    print("⏳ [TODO] Complete Sections 3, 4, and 5 first before enforcing Silver schemas.")


### 6.2 Solution — Sauvegarder les 6 Parquets Silver finaux (`silver/*.parquet`)
**L'objectif :** Exportez les 5 tables Silver typées et `quarantine_errors.parquet` dans `SILVER_OUT` en compression Snappy avec `index=False` (évitant la fuite `__index_level_0__`).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.to_parquet(chemin, index=False, compression="snappy")` | Écriture Parquet Snappy sans colonne d'index | `df_out.to_parquet(SILVER_OUT / fname, index=False, compression="snappy")` |

In [ ]:
silver_outputs = {}
if schema_ready:
    silver_outputs = {
        "silver_telemetry.parquet": df_telem,
        "silver_stoppages.parquet": df_stop,
        "silver_lines.parquet": df_lines,
        "silver_shifts.parquet": df_shifts,
        "silver_bottle_formats.parquet": df_formats,
    }
    for fname, df_out in silver_outputs.items():
        df_out.to_parquet(SILVER_OUT / fname, index=False, compression="snappy")
        print(f"  [SAVED] {fname:<32} ({len(df_out):>6,} rows)")
    quarantine_df.to_parquet(SILVER_OUT / "quarantine_errors.parquet", index=False, compression="snappy")
    print(f"  [SAVED] {'quarantine_errors.parquet':<32} ({len(quarantine_df):>6,} audited DQ events)")
else:
    print("⏳ [TODO] Complete Step 6.1 before saving the final Silver Parquets.")


### 6.3 Audit — Comparaison Bronze vs Silver Final (Schémas Parquet, Footer & DuckDB `DESCRIBE`)
**Validation finale :** Comparez le schéma Arrow embarqué, le `physical_type` Parquet et les statistiques `null_count / min / max` du footer entre Bronze brut et Silver final — prouvant directement depuis les métadonnées Parquet qu'il n'y a plus aucun `NaN`, que la pression est dans `[0.0, 12.0] bar`, que `LINE_Z` et `2099` ont disparu et que DuckDB lit nativement `TIMESTAMP_NS`, `TIMESTAMPTZ_NS`, `DATE` et `BOOLEAN` !

In [ ]:
if silver_outputs:
    pf_bronze = pq.ParquetFile(BRONZE_STAGED / "bronze_telemetry.parquet")
    pf_silver = pq.ParquetFile(SILVER_OUT / "silver_telemetry.parquet")
    bronze_types = {f.name: str(f.type) for f in pf_bronze.schema_arrow}
    silver_types = {f.name: str(f.type) for f in pf_silver.schema_arrow}
    rg_s = pf_silver.metadata.row_group(0)
    silver_stats = {rg_s.column(i).path_in_schema: rg_s.column(i) for i in range(rg_s.num_columns)}

    comparison = pd.DataFrame([
        {
            "column": col,
            "bronze_type": bronze_types.get(col, "<new>"),
            "silver_type": silver_types[col],
            "phys_type": silver_stats[col].physical_type,
            "nulls": silver_stats[col].statistics.null_count if silver_stats[col].statistics else None,
            "footer_min": repr(silver_stats[col].statistics.min) if silver_stats[col].statistics else None,
            "footer_max": repr(silver_stats[col].statistics.max) if silver_stats[col].statistics else None,
        }
        for col in silver_types
    ])
    print("=== BRONZE vs FINAL SILVER PARQUET SCHEMA & FOOTER STATS ===")
    print(comparison.to_string(index=False))

    # Assert strict Arrow/Parquet temporal & date contracts across ALL 5 saved Silver tables
    for fname in silver_outputs:
        sc = pq.ParquetFile(SILVER_OUT / fname).schema_arrow
        assert "__index_level_0__" not in set(sc.names), f"Index leak in {fname}!"
        ing_t = sc.field("_ingestion_timestamp_utc").type
        assert pa.types.is_timestamp(ing_t) and ing_t.tz == "UTC", f"{fname}: expected UTC timestamp, got {ing_t}"

    assert pa.types.is_timestamp(pf_silver.schema_arrow.field("timestamp").type)
    assert pa.types.is_date(pf_silver.schema_arrow.field("operational_shift_date").type)
    stop_sc = pq.ParquetFile(SILVER_OUT / "silver_stoppages.parquet").schema_arrow
    assert pa.types.is_timestamp(stop_sc.field("event_date").type)
    assert pa.types.is_date(stop_sc.field("operational_shift_date").type)

    # Verify DuckDB sees native TIMESTAMP_NS / TIMESTAMPTZ_NS / DATE / BOOLEAN types directly from Parquet
    con = duckdb.connect()
    duck_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SILVER_OUT / 'silver_telemetry.parquet'}')").df()
    con.close()
    print("\nDuckDB DESCRIBE on silver_telemetry.parquet:")
    print(duck_schema[["column_name", "column_type"]].to_string(index=False))
    print(f"\n[PASS] Complete Silver Layer ready for Lab 05 (DuckDB Star Schema & Joins) -> {SILVER_OUT}")
else:
    print("⏳ [TODO] Run Steps 6.1 and 6.2 first to audit the final Silver Parquets.")
